# Trabajo Práctico Final — Clasificación de Emociones
## Paso 3: ML Clásico — Extracción de Features + XGBoost + RFECV

**Estrategia:**
1. Usar la CNN del Paso 2 como **extractor de features** (encoder sin cabeza de clasificación)
2. Entrenar **XGBoost** (Gradient Boosting sobre árboles de decisión) sobre esos vectores
3. Aplicar **RFECV** (Recursive Feature Elimination con Cross-Validation) para identificar las features más informativas
4. Comparar el XGBoost-sobre-features vs el clasificador directo de la CNN

---

## 3.0 — Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.feature_selection import RFECV
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

try:
    from xgboost import XGBClassifier
    XGBOOST_OK = True
except ImportError:
    print('XGBoost no instalado — instalá con: pip install xgboost')
    XGBOOST_OK = False

SEED = 42
np.random.seed(SEED)

BASE_DIR = Path(r'C:\Users\Uriel\Documents\UTN\PIYS')
CLASSES  = ['angry','disgusted','fearful','happy','neutral','sad','surprised']

print('Imports OK')

## 3.1 — Carga de features extraídas en el Paso 2

In [ ]:
# Features de MiniEmotionNet (256 dims)
X_train_cnn = np.load(BASE_DIR / 'feat_train_cnn.npy')
y_train_cnn = np.load(BASE_DIR / 'feat_train_labels_cnn.npy')
X_test_cnn  = np.load(BASE_DIR / 'feat_test_cnn.npy')
y_test_cnn  = np.load(BASE_DIR / 'feat_test_labels_cnn.npy')

# Features de ResNet18 (512 dims)
X_train_rn = np.load(BASE_DIR / 'feat_train_rn.npy')
y_train_rn = np.load(BASE_DIR / 'feat_train_labels_rn.npy')
X_test_rn  = np.load(BASE_DIR / 'feat_test_rn.npy')
y_test_rn  = np.load(BASE_DIR / 'feat_test_labels_rn.npy')

print(f'CNN   features — Train: {X_train_cnn.shape}, Test: {X_test_cnn.shape}')
print(f'ResNet features — Train: {X_train_rn.shape},  Test: {X_test_rn.shape}')

# Verificación de distribución de clases
unique, counts = np.unique(y_train_cnn, return_counts=True)
for cls_id, cnt in zip(unique, counts):
    print(f'  Clase {CLASSES[cls_id]:12s}: {cnt:,}')

## 3.2 — Estandarización de features

El XGBoost es robusto al escalado, pero la estandarización ayuda a la convergencia y al análisis de importancias.

In [ ]:
scaler_cnn = StandardScaler()
X_train_cnn_sc = scaler_cnn.fit_transform(X_train_cnn)
X_test_cnn_sc  = scaler_cnn.transform(X_test_cnn)

scaler_rn = StandardScaler()
X_train_rn_sc = scaler_rn.fit_transform(X_train_rn)
X_test_rn_sc  = scaler_rn.transform(X_test_rn)

print(f'CNN scaled  — mean≈{X_train_cnn_sc.mean():.3f}, std≈{X_train_cnn_sc.std():.3f}')
print(f'ResNet scaled — mean≈{X_train_rn_sc.mean():.3f}, std≈{X_train_rn_sc.std():.3f}')

## 3.3 — XGBoost sobre features de MiniEmotionNet

In [ ]:
if XGBOOST_OK:
    xgb_cnn = XGBClassifier(
        n_estimators=500,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=20,
    )

    # Split interno para early stopping
    from sklearn.model_selection import train_test_split
    Xtr, Xval, ytr, yval = train_test_split(
        X_train_cnn_sc, y_train_cnn, test_size=0.15,
        stratify=y_train_cnn, random_state=SEED
    )

    print('Entrenando XGBoost sobre features CNN...')
    xgb_cnn.fit(
        Xtr, ytr,
        eval_set=[(Xval, yval)],
        verbose=50
    )

    y_pred_xgb_cnn = xgb_cnn.predict(X_test_cnn_sc)
    acc_xgb_cnn    = accuracy_score(y_test_cnn, y_pred_xgb_cnn)
    print(f'\nXGBoost-CNN Test Accuracy: {acc_xgb_cnn:.4f}')
    print(classification_report(y_test_cnn, y_pred_xgb_cnn, target_names=CLASSES))

    np.save(BASE_DIR / 'xgb_cnn_y_pred.npy', y_pred_xgb_cnn)
    y_prob_xgb_cnn = xgb_cnn.predict_proba(X_test_cnn_sc)
    np.save(BASE_DIR / 'xgb_cnn_y_prob.npy', y_prob_xgb_cnn)
    np.save(BASE_DIR / 'xgb_cnn_y_true.npy', y_test_cnn)

## 3.4 — XGBoost sobre features de ResNet18

In [ ]:
if XGBOOST_OK:
    xgb_rn = XGBClassifier(
        n_estimators=500,
        max_depth=6,
        learning_rate=0.04,
        subsample=0.8,
        colsample_bytree=0.7,
        use_label_encoder=False,
        eval_metric='mlogloss',
        random_state=SEED,
        n_jobs=-1,
        early_stopping_rounds=20,
    )

    Xtr2, Xval2, ytr2, yval2 = train_test_split(
        X_train_rn_sc, y_train_rn, test_size=0.15,
        stratify=y_train_rn, random_state=SEED
    )

    print('Entrenando XGBoost sobre features ResNet18...')
    xgb_rn.fit(Xtr2, ytr2, eval_set=[(Xval2, yval2)], verbose=50)

    y_pred_xgb_rn = xgb_rn.predict(X_test_rn_sc)
    acc_xgb_rn    = accuracy_score(y_test_rn, y_pred_xgb_rn)
    print(f'\nXGBoost-ResNet Test Accuracy: {acc_xgb_rn:.4f}')
    print(classification_report(y_test_rn, y_pred_xgb_rn, target_names=CLASSES))

    y_prob_xgb_rn = xgb_rn.predict_proba(X_test_rn_sc)
    np.save(BASE_DIR / 'xgb_rn_y_pred.npy', y_pred_xgb_rn)
    np.save(BASE_DIR / 'xgb_rn_y_prob.npy', y_prob_xgb_rn)
    np.save(BASE_DIR / 'xgb_rn_y_true.npy', y_test_rn)

## 3.5 — Feature Importance del XGBoost

XGBoost calcula la importancia de cada feature según cuánto reduce la impureza (Gain) al ser usada en una división. Las features del encoder son dimensiones del espacio latente aprendido por la CNN.

In [ ]:
if XGBOOST_OK:
    importances = xgb_cnn.feature_importances_
    feat_names  = [f'CNN_feat_{i}' for i in range(len(importances))]

    # Top 30 más importantes
    top_k = 30
    top_idx = np.argsort(importances)[::-1][:top_k]

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Barplot top-30
    ax = axes[0]
    colors = plt.cm.viridis(np.linspace(0.2, 0.9, top_k))
    ax.barh(range(top_k), importances[top_idx][::-1], color=colors[::-1])
    ax.set_yticks(range(top_k))
    ax.set_yticklabels([feat_names[i] for i in top_idx[::-1]], fontsize=7)
    ax.set_xlabel('Gain (importancia)')
    ax.set_title(f'Top {top_k} features — XGBoost sobre CNN encoder', fontweight='bold')
    ax.grid(axis='x', alpha=0.3)

    # Distribución acumulada de importancias
    ax = axes[1]
    sorted_imp = np.sort(importances)[::-1]
    cumsum     = np.cumsum(sorted_imp) / sorted_imp.sum() * 100
    ax.plot(cumsum, color='steelblue', lw=2)
    for pct in [80, 90, 95]:
        n_feats = np.searchsorted(cumsum, pct) + 1
        ax.axhline(pct, color='gray', lw=1, linestyle='--')
        ax.axvline(n_feats, color='red', lw=1, linestyle=':')
        ax.annotate(f'{pct}% con {n_feats} features',
                    xy=(n_feats, pct), xytext=(n_feats+5, pct-5), fontsize=8)
    ax.set_xlabel('Número de features (ordenadas por importancia)')
    ax.set_ylabel('Importancia acumulada (%)')
    ax.set_title('Curva de importancia acumulada', fontweight='bold')
    ax.grid(alpha=0.3)

    plt.suptitle('Análisis de Feature Importance — XGBoost', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.savefig(BASE_DIR / 'fig_10_feature_importance_xgb.png', dpi=150, bbox_inches='tight')
    plt.show()

    n_80 = np.searchsorted(cumsum, 80) + 1
    print(f'→ El {100*n_80/len(importances):.1f}% de las features ({n_80}/{len(importances)}) acumula el 80% de la importancia.')

## 3.6 — Selección de features con RFECV

**RFECV (Recursive Feature Elimination with Cross-Validation):** elimina iterativamente las features menos importantes y evalúa con CV el rendimiento resultante. El objetivo es encontrar el **subconjunto mínimo** de features que mantiene el accuracy del modelo completo.

Usamos un estimador `LogisticRegression` como proxy (más rápido que XGBoost para RFECV).

In [ ]:
# RFECV puede ser costoso con 256 features; muestreamos para rapidez
MAX_SAMPLES = 8000
if len(X_train_cnn_sc) > MAX_SAMPLES:
    idx = np.random.choice(len(X_train_cnn_sc), MAX_SAMPLES, replace=False)
    X_rfe = X_train_cnn_sc[idx]
    y_rfe = y_train_cnn[idx]
else:
    X_rfe, y_rfe = X_train_cnn_sc, y_train_cnn

estimator_rfe = LogisticRegression(
    max_iter=300, solver='saga', multi_class='multinomial',
    C=1.0, random_state=SEED, n_jobs=-1
)

rfecv = RFECV(
    estimator=estimator_rfe,
    step=10,                    # eliminar 10 features por iteración
    cv=StratifiedKFold(5),
    scoring='accuracy',
    min_features_to_select=20,
    n_jobs=-1,
    verbose=1,
)

print(f'Ejecutando RFECV sobre {X_rfe.shape[0]} muestras, {X_rfe.shape[1]} features...')
rfecv.fit(X_rfe, y_rfe)

print(f'\n→ Features óptimas: {rfecv.n_features_}')
print(f'→ Accuracy CV con features óptimas: {rfecv.cv_results_["mean_test_score"].max():.4f}')

In [ ]:
# Curva RFECV
n_feats_range = range(
    rfecv.min_features_to_select,
    X_train_cnn_sc.shape[1] + 1,
    rfecv.step
)
mean_scores = rfecv.cv_results_['mean_test_score']
std_scores  = rfecv.cv_results_['std_test_score']

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(list(n_feats_range)[:len(mean_scores)], mean_scores, 'o-', color='steelblue', ms=4, lw=2)
ax.fill_between(
    list(n_feats_range)[:len(mean_scores)],
    mean_scores - std_scores,
    mean_scores + std_scores,
    alpha=0.2, color='steelblue'
)
ax.axvline(rfecv.n_features_, color='red', lw=2, linestyle='--',
           label=f'Óptimo: {rfecv.n_features_} features')
ax.set_xlabel('Número de features')
ax.set_ylabel('Accuracy (CV 5-fold)')
ax.set_title('RFECV — Curva de selección de features\nMiniEmotionNet encoder → LogisticRegression',
             fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(BASE_DIR / 'fig_11_rfecv_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 3.7 — XGBoost con subconjunto de features seleccionadas

In [ ]:
if XGBOOST_OK:
    selected_mask = rfecv.support_
    X_train_sel = X_train_cnn_sc[:, selected_mask]
    X_test_sel  = X_test_cnn_sc[:, selected_mask]

    print(f'Entrenando XGBoost con {selected_mask.sum()} features seleccionadas...')

    xgb_sel = XGBClassifier(
        n_estimators=500, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric='mlogloss',
        random_state=SEED, n_jobs=-1, early_stopping_rounds=20
    )

    Xtr_s, Xval_s, ytr_s, yval_s = train_test_split(
        X_train_sel, y_train_cnn, test_size=0.15,
        stratify=y_train_cnn, random_state=SEED
    )
    xgb_sel.fit(Xtr_s, ytr_s, eval_set=[(Xval_s, yval_s)], verbose=50)

    y_pred_sel = xgb_sel.predict(X_test_sel)
    acc_sel    = accuracy_score(y_test_cnn, y_pred_sel)
    print(f'\nXGBoost ({selected_mask.sum()} features) Test Accuracy: {acc_sel:.4f}')
    print(f'XGBoost (todas {len(selected_mask)} features) Test Accuracy: {acc_xgb_cnn:.4f}')
    reduction = (1 - selected_mask.sum()/len(selected_mask)) * 100
    print(f'Reducción de features: {reduction:.1f}% con pérdida mínima de accuracy')

    y_prob_sel = xgb_sel.predict_proba(X_test_sel)
    np.save(BASE_DIR / 'xgb_sel_y_pred.npy', y_pred_sel)
    np.save(BASE_DIR / 'xgb_sel_y_prob.npy', y_prob_sel)
    np.save(BASE_DIR / 'xgb_sel_y_true.npy', y_test_cnn)
    np.save(BASE_DIR / 'rfecv_support_mask.npy', selected_mask)

## 3.8 — Tabla comparativa de todos los modelos (preview)

In [ ]:
from sklearn.metrics import accuracy_score

results = []

# CNN directo
y_pred_cnn_direct = np.load(BASE_DIR / 'cnn_y_pred.npy')
y_true_cnn_direct = np.load(BASE_DIR / 'cnn_y_true.npy')
results.append({'Modelo': 'MiniEmotionNet (CNN directa)',
                'Acc Test': accuracy_score(y_true_cnn_direct, y_pred_cnn_direct),
                'Features': 'N/A (end-to-end)'})

# ResNet18 directo
y_pred_rn_direct = np.load(BASE_DIR / 'resnet_y_pred.npy')
y_true_rn_direct = np.load(BASE_DIR / 'resnet_y_true.npy')
results.append({'Modelo': 'ResNet18 Transfer Learning',
                'Acc Test': accuracy_score(y_true_rn_direct, y_pred_rn_direct),
                'Features': 'N/A (end-to-end)'})

if XGBOOST_OK:
    results.append({'Modelo': 'XGBoost + CNN features (256)',
                    'Acc Test': acc_xgb_cnn, 'Features': '256'})
    results.append({'Modelo': 'XGBoost + ResNet features (512)',
                    'Acc Test': acc_xgb_rn,  'Features': '512'})
    results.append({'Modelo': f'XGBoost + RFECV ({selected_mask.sum()} features)',
                    'Acc Test': acc_sel,      'Features': str(selected_mask.sum())})

df_results = pd.DataFrame(results).sort_values('Acc Test', ascending=False)
df_results['Acc Test'] = df_results['Acc Test'].round(4)
df_results.to_csv(BASE_DIR / 'model_comparison.csv', index=False)
print(df_results.to_string(index=False))

## Resumen del Paso 3

- Entrenamos **2 XGBoost** (sobre features CNN y ResNet18) y **1 XGBoost-RFECV** (features seleccionadas)
- RFECV identificó el subconjunto óptimo de features del espacio latente de la CNN
- Los artefactos `.npy` con predicciones y probabilidades están listos para el Paso 4 (ROC, Matrices de Confusión)